# Video Studio — `musetalk` on Colab (manual backup)

Kaggle is the studio's automated GPU provider. Use this notebook only when Kaggle has no GPU free.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Stay on the page.
2. When it finishes, the browser downloads **`vs-result-musetalk.zip`**. Give that file to Claude
   (upload it to the session or commit it to the branch); Claude runs `vs-colab import`.
3. The last cell releases the GPU so your Colab allowance is not wasted.

Colab rules: run it yourself in this notebook UI; no SSH, remote desktop or web UIs; one account;
no deepfakes — lip sync only on faces you own, AI-generated people, or people who agreed in writing.
Free Colab limits are not published and change; a run can be stopped if you are idle.

Same code as the Kaggle job (packed by `vs-colab build` from `gpu-jobs/`); models are downloaded
from their official Hugging Face / GitHub sources at pinned revisions.

In [ ]:
# 1. GPU check + map the Kaggle paths the job uses onto /content
import os, subprocess, pathlib
r = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
assert r.returncode == 0, 'No GPU: Runtime > Change runtime type > T4 GPU, then Run all again'
print(r.stdout)
for d in ('/content/kaggle/working', '/content/kaggle/temp'):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
if not os.path.exists('/kaggle'):
    os.symlink('/content/kaggle', '/kaggle')


In [ ]:
# 2. upload the inputs that are not embedded: presenter.png
import pathlib, shutil
from google.colab import files
need = ['presenter.png']
inp = pathlib.Path('/kaggle/working/in'); inp.mkdir(parents=True, exist_ok=True)
missing = [n for n in need if not (inp / n).exists()]
while missing:
    print('Upload:', ', '.join(missing))
    for fn, data in files.upload().items():
        base = pathlib.Path(fn).name
        target = base if base in need else (missing[0] if len(missing) == 1 else None)
        if target:
            (inp / target).write_bytes(data); print('saved', target)
    missing = [n for n in need if not (inp / n).exists()]


In [ ]:
# 3. the studio job (generated — do not edit; rebuild with vs-colab build)
JOB = '# --- packed by vs-gpu: inputs, params, shared runtime (gpu-jobs/_common.py), then the job ---\nimport base64, json, os, pathlib, time\n_T0 = time.time()\nIN = pathlib.Path("/kaggle/working/in"); OUT = pathlib.Path("/kaggle/working/out")\nIN.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)\nPARAMS = json.loads(\'{"items": [{"id": "talk", "video": "presenter.png", "audio": "chatterbox-ar-calm.wav"}], "max_job_s": 2400, "_need_gpu": true, "_need_internet": true, "parallel": false}\')\nfor _n, _b in json.loads(\'{"chatterbox-ar-calm.wav": "..."}\').items():\n    (IN / _n).write_bytes(base64.b64decode(_b))\n_COMMON_B64 = \'IyBTaGFyZWQgcnVudGltZSBmb3IgZXZlcnkgc3R1ZGlvIEdQVSBqb2IgKHBhY2tlZCBieSB2cy1ncHU7IHJ1bnMgaW5zaWRlIHRoZSBLYWdnbGUga2VybmVsIGFuZCBpdHMKIyBwZXItR1BVIGNoaWxkIHByb2Nlc3NlcykuIFByb3ZpZGVzOiByZXBvcnQoKSwgZ3B1X2luZm8oKSwgaXNfdDQoKSwgcGlja19kdHlwZSgpLCBzaGFyZCgpLCBsb2FkX3BpcGUoKS4KIyBLYWdnbGUgZnJlZSB0aWVyID0gMnggVGVzbGEgVDQgKHNtNzUsIDE0LjYgR0IgZWFjaCk6IGZhc3QgZnAxNiB0ZW5zb3IgY29yZXMsIGJmMTYgb25seSBlbXVsYXRlZC4KaW1wb3J0IGpzb24sIG9zLCBwYXRobGliLCBzdWJwcm9jZXNzLCBzeXMsIHRpbWUKClNIQVJEID0gaW50KG9zLmVudmlyb24uZ2V0KCJWU19TSEFSRCIsICItMSIpKSAgICAgICAgICAjID49IDAgaW5zaWRlIGEgcGVyLUdQVSBjaGlsZCBwcm9jZXNzCgoKZGVmIHJlcG9ydCgqKmt3KToKICAgIHAgPSBPVVQgLyAoInJlcG9ydC5qc29uIiBpZiBTSEFSRCA8IDAgZWxzZSBmInJlcG9ydC5zaGFyZHtTSEFSRH0uanNvbiIpCiAgICBkID0ganNvbi5sb2FkcyhwLnJlYWRfdGV4dCgpKSBpZiBwLmV4aXN0cygpIGVsc2Uge30KICAgIGQudXBkYXRlKGt3KTsgZFsiZWxhcHNlZF9zIl0gPSByb3VuZCh0aW1lLnRpbWUoKSAtIF9UMCwgMSk7IHAud3JpdGVfdGV4dChqc29uLmR1bXBzKGQsIGluZGVudD0yKSkKCgpkZWYgX3ZyYW1fcGVhaygpOiAgICAgICAgICAgIyBiZW5jaG1hcms6IHBlYWsgVlJBTSBwZXIgdmlzaWJsZSBHUFUsIHdyaXR0ZW4gYXQgZXhpdCBvZiBldmVyeSAoY2hpbGQpIHByb2Nlc3MKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2gKICAgICAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICByZXBvcnQoKip7InZyYW1fcGVha19nYiI6IFtyb3VuZCh0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKGkpIC8gMioqMzAsIDIpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBpIGluIHJhbmdlKHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCkpXX0pCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MKCgpkZWYgX3dhdGNoZG9nKGxpbWl0X3MpOiAgICAjIGhhcmQgam9iIGJ1ZGdldDogbmV2ZXIgbGV0IGEgc3R1Y2sgam9iIGJ1cm4gR1BVIHF1b3RhIChLYWdnbGUgYWxsb3dzIDEyIGgpCiAgICBpbXBvcnQgdGhyZWFkaW5nCiAgICBkZWYgZmlyZSgpOgogICAgICAgIHJlcG9ydChlcnJvcj1mImpvYiBleGNlZWRlZCBpdHMgYnVkZ2V0IG9mIHtsaW1pdF9zfSBzIGFuZCB3YXMgc3RvcHBlZCAocmFpc2UgbWF4X2pvYl9zIG9ubHkgaWYgbmVlZGVkKSIsCiAgICAgICAgICAgICAgIHRpbWVkX291dD1UcnVlKQogICAgICAgIF92cmFtX3BlYWsoKTsgb3MuX2V4aXQoNSkKICAgIHQgPSB0aHJlYWRpbmcuVGltZXIobGltaXRfcywgZmlyZSk7IHQuZGFlbW9uID0gVHJ1ZTsgdC5zdGFydCgpCgoKaW1wb3J0IGF0ZXhpdAphdGV4aXQucmVnaXN0ZXIoX3ZyYW1fcGVhaykKaWYgU0hBUkQgPCAwIGFuZCBub3Qgb3MuZW52aXJvbi5nZXQoIlZTX0ZBS0VfR1BVUyIpOgogICAgX3dhdGNoZG9nKGludChQQVJBTVMuZ2V0KCJtYXhfam9iX3MiLCAzNjAwKSkpCgoKZGVmIHN0ZXBfZ3VhcmQodG90YWxfc3RlcHMsIGJ1ZGdldF9zLCBsYWJlbD0iaXRlbSIsIHdlaWdodHM9Tm9uZSk6CiAgICAiIiJkaWZmdXNlcnMgY2FsbGJhY2tfb25fc3RlcF9lbmQ6IHN0b3Agb24gTmFOL2luZiBsYXRlbnRzIGFuZCB3aGVuIHRoZSBwcm9qZWN0ZWQgdGltZSBvZiB0aGlzIGl0ZW0gZXhjZWVkcwogICAgYnVkZ2V0X3MgKGVhcmx5IGZhaWx1cmUgZGV0ZWN0aW9uLCBtZWFzdXJlZCBwZXIgc3RlcCkuIHdlaWdodHMgPSByZWxhdGl2ZSBjb3N0IHBlciBzdGVwICgyIHdpdGggQ0ZHLCAxCiAgICB3aXRob3V0KSBzbyB0aGUgcHJvamVjdGlvbiBzdGF5cyByaWdodCB3aGVuIENGRyBpcyBzd2l0Y2hlZCBvZmYgcGFydC13YXkuIFJhaXNlcyBGbG9hdGluZ1BvaW50RXJyb3IgLwogICAgVGltZW91dEVycm9yLiIiIgogICAgdDAgPSB0aW1lLnRpbWUoKQogICAgdyA9IGxpc3Qod2VpZ2h0cyBvciBbMV0gKiB0b3RhbF9zdGVwcykKICAgIHN0YXRzID0geyJzdGVwcyI6IDAsICJkZW5vaXNlX3MiOiAwLjB9CiAgICBkZWYgY2IocGlwZSwgc3RlcCwgdHMsIGt3KToKICAgICAgICBpbXBvcnQgdG9yY2gKICAgICAgICBsYXQgPSBrdy5nZXQoImxhdGVudHMiKQogICAgICAgIGlmIGxhdCBpcyBub3QgTm9uZSBhbmQgbm90IHRvcmNoLmlzZmluaXRlKGxhdCkuYWxsKCk6CiAgICAgICAgICAgIHJhaXNlIEZsb2F0aW5nUG9pbnRFcnJvcihmIntsYWJlbH06IG5vbi1maW5pdGUgbGF0ZW50cyBhdCBzdGVwIHtzdGVwfSIpCiAgICAgICAgZWwgPSB0aW1lLnRpbWUoKSAtIHQwCiAgICAgICAgcHJvaiA9IGVsIC8gc3VtKHdbOnN0ZXAgKyAxXSkgKiBzdW0odykKICAgICAgICBpZiBzdGVwID49IDEgYW5kIHByb2ogPiBidWRnZXRfczoKICAgICAgICAgICAgcmFpc2UgVGltZW91dEVycm9yKGYie2xhYmVsfToge2VsIC8gKHN0ZXAgKyAxKTouMWZ9IHMvc3RlcCwgcHJvamVjdGVkIHtwcm9qOi4wZn0gcyA+IGJ1ZGdldCB7YnVkZ2V0X3N9IHMiKQogICAgICAgIHN0YXRzLnVwZGF0ZShzdGVwcz1zdGVwICsgMSwgZGVub2lzZV9zPXJvdW5kKGVsLCAxKSkKICAgICAgICByZXR1cm4ga3cKICAgIGNiLnN0YXRzID0gc3RhdHMKICAgIHJldHVybiBjYgoKCmRlZiBncHVfaW5mbygpOgogICAgaWYgb3MuZW52aXJvbi5nZXQoIlZTX0ZBS0VfR1BVUyIpOiAgICAgICAjIENQVSBzZWxmLXRlc3Qgb2YgcGFja2luZyBhbmQgZmFuLW91dCAodGVzdHMvdGVzdF9ncHVfcnVudGltZS5zaCkKICAgICAgICByZXR1cm4geyJuIjogaW50KG9zLmVudmlyb25bIlZTX0ZBS0VfR1BVUyJdKSwgIm5hbWUiOiAiZmFrZSIsICJ2cmFtX2diIjogMTQuNiwgImNjIjogNy41fQogICAgaW1wb3J0IHRvcmNoCiAgICBuID0gdG9yY2guY3VkYS5kZXZpY2VfY291bnQoKQogICAgaWYgbm90IG46CiAgICAgICAgcmV0dXJuIHsibiI6IDB9CiAgICBwID0gdG9yY2guY3VkYS5nZXRfZGV2aWNlX3Byb3BlcnRpZXMoMCkKICAgIHJldHVybiB7Im4iOiBuLCAibmFtZSI6IHAubmFtZSwgInZyYW1fZ2IiOiByb3VuZChwLnRvdGFsX21lbW9yeSAvIDIqKjMwLCAxKSwgImNjIjogcC5tYWpvciArIHAubWlub3IgLyAxMH0KCgpkZWYgaXNfdDQoKToKICAgIGcgPSBncHVfaW5mbygpCiAgICByZXR1cm4gZy5nZXQoIm4iLCAwKSA+IDAgYW5kIGdbImNjIl0gPCA4LjAKCgpkZWYgcGlja19kdHlwZShmcDE2X3NhZmUpOgogICAgIiIiZnAxNiBvbiBUNCBmb3IgbW9kZWxzIGtub3duIHRvIGJlIGZwMTYtc2FmZSAoNC04eCBmYXN0ZXIgdGhhbiBlbXVsYXRlZCBiZjE2KTsgYmYxNiBvdGhlcndpc2UuCiAgICBQQVJBTVNbImR0eXBlIl0gPSBmcDE2IHwgYmYxNiB8IGZwMzIgb3ZlcnJpZGVzLiIiIgogICAgaW1wb3J0IHRvcmNoCiAgICB3YW50ID0gUEFSQU1TLmdldCgiZHR5cGUiLCAiYXV0byIpCiAgICBpZiB3YW50IGluICgiZnAxNiIsICJiZjE2IiwgImZwMzIiKToKICAgICAgICByZXR1cm4geyJmcDE2IjogdG9yY2guZmxvYXQxNiwgImJmMTYiOiB0b3JjaC5iZmxvYXQxNiwgImZwMzIiOiB0b3JjaC5mbG9hdDMyfVt3YW50XQogICAgaWYgaXNfdDQoKToKICAgICAgICByZXR1cm4gdG9yY2guZmxvYXQxNiBpZiBmcDE2X3NhZmUgZWxzZSB0b3JjaC5iZmxvYXQxNgogICAgcmV0dXJuIHRvcmNoLmJmbG9hdDE2CgoKY2xhc3MgX0RvbmUoRXhjZXB0aW9uKToKICAgIHBhc3MKCgpkZWYgc2hhcmQoaXRlbXMpOgogICAgIiIiU21hbGwgbW9kZWxzOiBvbmUgcHJvY2VzcyBwZXIgR1BVLCBpdGVtcyBzcGxpdCByb3VuZC1yb2JpbiAoMnggdGhyb3VnaHB1dCBvbiAyeCBUNCkuCiAgICBSZXR1cm5zIHRoaXMgcHJvY2VzcydzIGl0ZW1zOyBpbiB0aGUgcGFyZW50IGl0IHdhaXRzIGZvciB0aGUgY2hpbGRyZW4sIG1lcmdlcyByZXBvcnRzIGFuZCBzdG9wcyB0aGUgam9iLiIiIgogICAgbiA9IGdwdV9pbmZvKCkuZ2V0KCJuIiwgMCkKICAgIGlmIFNIQVJEID49IDA6CiAgICAgICAgcmV0dXJuIGl0ZW1zW1NIQVJEOjppbnQob3MuZW52aXJvblsiVlNfTlNIQVJEIl0pXQogICAgaWYgbiA8IDIgb3IgbGVuKGl0ZW1zKSA8IDIgb3Igbm90IFBBUkFNUy5nZXQoInBhcmFsbGVsIiwgVHJ1ZSk6CiAgICAgICAgcmV0dXJuIGl0ZW1zCiAgICB0bXAgPSBwYXRobGliLlBhdGgoIi9rYWdnbGUvdGVtcCIpOyB0bXAubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgKHRtcCAvICJ2c19wYXJhbXMuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhQQVJBTVMpKQogICAgKHRtcCAvICJ2c19jaGlsZC5weSIpLndyaXRlX3RleHQoCiAgICAgICAgImltcG9ydCBiYXNlNjQsIGpzb24sIHBhdGhsaWIsIHRpbWVcbl9UMCA9IHRpbWUudGltZSgpXG4iCiAgICAgICAgIklOID0gcGF0aGxpYi5QYXRoKCcva2FnZ2xlL3dvcmtpbmcvaW4nKTsgT1VUID0gcGF0aGxpYi5QYXRoKCcva2FnZ2xlL3dvcmtpbmcvb3V0JylcbiIKICAgICAgICAiUEFSQU1TID0ganNvbi5sb2FkKG9wZW4oJy9rYWdnbGUvdGVtcC92c19wYXJhbXMuanNvbicpKVxuIgogICAgICAgIGYiX0NPTU1PTl9CNjQgPSB7X0NPTU1PTl9CNjQhcn1cbl9KT0JfQjY0ID0ge19KT0JfQjY0IXJ9XG4iCiAgICAgICAgImV4ZWMoYmFzZTY0LmI2NGRlY29kZShfQ09NTU9OX0I2NCkuZGVjb2RlKCkpXG4iCiAgICAgICAgImV4ZWMoY29tcGlsZShiYXNlNjQuYjY0ZGVjb2RlKF9KT0JfQjY0KS5kZWNvZGUoKSwgJ2pvYi5weScsICdleGVjJykpXG4iKQogICAgcHJvY3MgPSBbc3VicHJvY2Vzcy5Qb3Blbihbc3lzLmV4ZWN1dGFibGUsIHN0cih0bXAgLyAidnNfY2hpbGQucHkiKV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGVudj17Kipvcy5lbnZpcm9uLCAiVlNfU0hBUkQiOiBzdHIoaSksICJWU19OU0hBUkQiOiBzdHIobiksICJDVURBX1ZJU0lCTEVfREVWSUNFUyI6IHN0cihpKX0pCiAgICAgICAgICAgICBmb3IgaSBpbiByYW5nZShuKV0KICAgIGNvZGVzID0gW3Aud2FpdCgpIGZvciBwIGluIHByb2NzXQogICAgc2hhcmRzID0gW10KICAgIGZvciBpIGluIHJhbmdlKG4pOgogICAgICAgIGYgPSBPVVQgLyBmInJlcG9ydC5zaGFyZHtpfS5qc29uIgogICAgICAgIGlmIGYuZXhpc3RzKCk6CiAgICAgICAgICAgIHNoYXJkcy5hcHBlbmQoanNvbi5sb2FkcyhmLnJlYWRfdGV4dCgpKSk7IGYudW5saW5rKCkKICAgIHJlcG9ydChwYXJhbGxlbF9ncHVzPW4sIHNoYXJkcz1zaGFyZHMsIHNoYXJkX2V4aXRfY29kZXM9Y29kZXMpCiAgICBpZiBjb2RlcyBhbmQgYWxsKGNvZGVzKToKICAgICAgICByZXBvcnQoZXJyb3I9ZiJldmVyeSBHUFUgd29ya2VyIGZhaWxlZDogZXhpdCBjb2RlcyB7Y29kZXN9IikKICAgIGVsaWYgYW55KGNvZGVzKTogICAgICAgICAgICAgICAgICAgICAgICAgICMgcGFydGlhbCByZXN1bHQ6IGtlZXAgd2hhdCB3YXMgbWFkZSwgdnMtcHJvZHVjZSBmYWxscyBiYWNrIGZvciB0aGUgcmVzdAogICAgICAgIHJlcG9ydCh3YXJuaW5nPWYiYSBHUFUgd29ya2VyIGZhaWxlZDogZXhpdCBjb2RlcyB7Y29kZXN9IikKICAgIHJhaXNlIF9Eb25lKCkKCgpkZWYgbG9hZF9waXBlKGNscywgcmVwbywgcmV2aXNpb24sIGR0eXBlLCBmcDhfb2s9VHJ1ZSwgcHJlZmVyPU5vbmUsICoqa3cpOgogICAgIiIiQmlnIGRpZmZ1c2lvbiBtb2RlbHMgb24gMnggVDQsIGJlc3Qgc3RyYXRlZ3kgZmlyc3Q6CiAgICAgICAxKSBkZXZpY2VfbWFwPSdiYWxhbmNlZCcg4oCUIHRleHQgZW5jb2RlciBvbiBvbmUgVDQsIHRyYW5zZm9ybWVyICsgVkFFIG9uIHRoZSBvdGhlciAobm8gQ1BVIHJvdW5kLXRyaXBzKQogICAgICAgMikgZnA4IHdlaWdodCBzdG9yYWdlIGZvciB0aGUgdHJhbnNmb3JtZXIgKyBtb2RlbCBDUFUgb2ZmbG9hZCBvbiBvbmUgVDQKICAgICAgIDMpIHNlcXVlbnRpYWwgQ1BVIG9mZmxvYWQgKHNsb3dlc3QsIGFsd2F5cyBmaXRzKQogICAgUEFSQU1TWyJwbGFjZW1lbnQiXSA9IGJhbGFuY2VkIHwgb2ZmbG9hZCB8IHNlcXVlbnRpYWwgZm9yY2VzIG9uZS4iIiIKICAgIGltcG9ydCB0b3JjaAogICAgZm9yY2UgPSBwcmVmZXIgb3IgUEFSQU1TLmdldCgicGxhY2VtZW50IikKICAgIGVycm9ycyA9IFtdCiAgICBpZiAoZm9yY2UgaW4gKE5vbmUsICJiYWxhbmNlZCIpKSBhbmQgZ3B1X2luZm8oKS5nZXQoIm4iLCAwKSA+PSAyOgogICAgICAgIHRyeToKICAgICAgICAgICAgcGlwZSA9IGNscy5mcm9tX3ByZXRyYWluZWQocmVwbywgcmV2aXNpb249cmV2aXNpb24sIHRvcmNoX2R0eXBlPWR0eXBlLCBkZXZpY2VfbWFwPSJiYWxhbmNlZCIsICoqa3cpCiAgICAgICAgICAgIGZwOCA9IGZwOF9vayBhbmQgZ3B1X2luZm8oKVsidnJhbV9nYiJdIDwgMjAgYW5kIGhhc2F0dHIocGlwZSwgInRyYW5zZm9ybWVyIikKICAgICAgICAgICAgaWYgZnA4OiAgICMgbWVhc3VyZWQ6IFotSW1hZ2UncyA2QiB0cmFuc2Zvcm1lciBhbG9uZSAoMTIgR0IgYmYxNikgKyBhdHRlbnRpb24gb3ZlcmZsb3dzIGEgMTQuNiBHQiBUNAogICAgICAgICAgICAgICAgcGlwZS50cmFuc2Zvcm1lci5lbmFibGVfbGF5ZXJ3aXNlX2Nhc3Rpbmcoc3RvcmFnZV9kdHlwZT10b3JjaC5mbG9hdDhfZTRtM2ZuLCBjb21wdXRlX2R0eXBlPWR0eXBlKQogICAgICAgICAgICAgICAgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCiAgICAgICAgICAgIHJlcG9ydChwbGFjZW1lbnQ9ImJhbGFuY2VkICgyIEdQVXMpIiArICgiICsgZnA4IHRyYW5zZm9ybWVyIHdlaWdodHMiIGlmIGZwOCBlbHNlICIiKSkKICAgICAgICAgICAgcmV0dXJuIHBpcGUKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJiYWxhbmNlZDoge3R5cGUoZSkuX19uYW1lX199OiB7c3RyKGUpWzoyMDBdfSIpCiAgICBpZiBmb3JjZSBpbiAoTm9uZSwgIm9mZmxvYWQiKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHBpcGUgPSBjbHMuZnJvbV9wcmV0cmFpbmVkKHJlcG8sIHJldmlzaW9uPXJldmlzaW9uLCB0b3JjaF9kdHlwZT1kdHlwZSwgKiprdykKICAgICAgICAgICAgaWYgZnA4X29rIGFuZCBncHVfaW5mbygpWyJ2cmFtX2diIl0gPCAyMCBhbmQgaGFzYXR0cihwaXBlLCAidHJhbnNmb3JtZXIiKToKICAgICAgICAgICAgICAgIHBpcGUudHJhbnNmb3JtZXIuZW5hYmxlX2xheWVyd2lzZV9jYXN0aW5nKHN0b3JhZ2VfZHR5cGU9dG9yY2guZmxvYXQ4X2U0bTNmbiwgY29tcHV0ZV9kdHlwZT1kdHlwZSkKICAgICAgICAgICAgcGlwZS5lbmFibGVfbW9kZWxfY3B1X29mZmxvYWQoZ3B1X2lkPTApCiAgICAgICAgICAgIHJlcG9ydChwbGFjZW1lbnQ9ImZwOCB0cmFuc2Zvcm1lciArIG1vZGVsIENQVSBvZmZsb2FkICgxIEdQVSkiLCBwbGFjZW1lbnRfZXJyb3JzPWVycm9ycykKICAgICAgICAgICAgcmV0dXJuIHBpcGUKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJvZmZsb2FkOiB7dHlwZShlKS5fX25hbWVfX306IHtzdHIoZSlbOjIwMF19IikKICAgIHBpcGUgPSBjbHMuZnJvbV9wcmV0cmFpbmVkKHJlcG8sIHJldmlzaW9uPXJldmlzaW9uLCB0b3JjaF9kdHlwZT1kdHlwZSwgKiprdykKICAgIHBpcGUuZW5hYmxlX3NlcXVlbnRpYWxfY3B1X29mZmxvYWQoKQogICAgcmVwb3J0KHBsYWNlbWVudD0ic2VxdWVudGlhbCBDUFUgb2ZmbG9hZCIsIHBsYWNlbWVudF9lcnJvcnM9ZXJyb3JzKQogICAgcmV0dXJuIHBpcGUKCgpkZWYgdmFlX3RpbGluZyhwaXBlKToKICAgIGZvciBuYW1lIGluICgidmFlIiwpOgogICAgICAgIG0gPSBnZXRhdHRyKHBpcGUsIG5hbWUsIE5vbmUpCiAgICAgICAgZm9yIGZuIGluICgiZW5hYmxlX3RpbGluZyIsICJlbmFibGVfc2xpY2luZyIpOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBnZXRhdHRyKG0sIGZuKSgpCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICBwYXNzCgoKZGVmIGxvb2tzX2Jyb2tlbihhcnIpOgogICAgIiIiTmFOL2JsYWNrIG91dHB1dCBjaGVjayAoZnAxNiBvdmVyZmxvdyBzaG93cyB1cCBhcyBibGFjayBmcmFtZXMpLiIiIgogICAgaW1wb3J0IG51bXB5IGFzIG5wCiAgICBhID0gbnAuYXNhcnJheShhcnIsIGR0eXBlPW5wLmZsb2F0MzIpCiAgICBpZiBub3QgbnAuaXNmaW5pdGUoYSkuYWxsKCk6CiAgICAgICAgcmV0dXJuIFRydWUKICAgIGlmIGEuc2l6ZSBhbmQgZmxvYXQoYS5tYXgoKSkgPD0gMS4wOiAgICAgICMgcGlwZWxpbmVzIHJldHVybmluZyBmbG9hdHMgaW4gWzAsIDFdIChlLmcuIFdhbiB2aWRlbyBmcmFtZXMpCiAgICAgICAgYSA9IGEgKiAyNTUuMAogICAgcmV0dXJuIGZsb2F0KGEubWVhbigpKSA8IDIuMAo=\'\n_JOB_B64 = \'IyBHUFUgam9iOiBsaXAtc3luYyBhIGZhY2UgdmlkZW8gb3Igc3RpbGwgdG8gc3BlZWNoIHdpdGggTXVzZVRhbGsgMS41IChUZW5jZW50IEx5cmEgTGFiLCBNSVQpLgojIFBBUkFNUzogeyJpdGVtcyI6W3siaWQiOiJzaG90MDMiLCJ2aWRlbyI6ImZhY2UubXA0IiBvciAiZmFjZS5wbmciLCJhdWRpbyI6InZvMy53YXYifV19ICAoZmlsZXMgaW4gSU4vKQojIFQ0OiBmcDE2IGluZmVyZW5jZSAoLS11c2VfZmxvYXQxNiksIHNpbmdsZSBHUFUgKH41IEdCKS4KIyBDb25zZW50IHJ1bGU6IG9ubHkgZmFjZXMgdGhlIHBlcnNvbiBvd25zLCBBSS1nZW5lcmF0ZWQgcGVvcGxlLCBvciBwZW9wbGUgd2hvIGFncmVlZCBpbiB3cml0aW5nLgojCiMgUmV2aWV3ZWQgMjAyNi0xMS0yNSBhdCBjb21taXQgMjUzNmVhZjogdXBzdHJlYW0gZG93bmxvYWRfd2VpZ2h0cy5zaCByb3V0ZXMgSHVnZ2luZyBGYWNlIHRocm91Z2ggYQojIHRoaXJkLXBhcnR5IG1pcnJvciAoaGYtbWlycm9yLmNvbSkgYW5kIHB1bGxzIGEgR29vZ2xlIERyaXZlIGZpbGU7IHJlcXVpcmVtZW50cy50eHQgcGlucyBUZW5zb3JGbG93IDIuMTIKIyBhbmQgb2xkIGRpZmZ1c2Vycy4gVGhpcyBqb2IgdGhlcmVmb3JlIGRvd25sb2FkcyBldmVyeSB3ZWlnaHQgaXRzZWxmIGZyb20gdGhlIG9mZmljaWFsIHNvdXJjZXMgYW5kIGluc3RhbGxzCiMgb25seSB3aGF0IGluZmVyZW5jZSBuZWVkcy4gV2VpZ2h0czogTXVzZVRhbGsgKE1JVCksIHNkLXZhZS1mdC1tc2UgKE1JVCksIHdoaXNwZXItdGlueSAoQXBhY2hlLTIuMCksCiMgZmFjZS1wYXJzaW5nIEJpU2VOZXQgKFdURlBMIHJlLXVwbG9hZCksIHJlc25ldDE4IChCU0QtMykuIFMzRkQgZmFjZSBkZXRlY3RvciAoQlNELTMsIHZpYSB0b3JjaCBodWIpLgppbXBvcnQgb3MsIHN1YnByb2Nlc3MsIHN5cywgdXJsbGliLnJlcXVlc3QKU1JDLCBSRUYgPSAiaHR0cHM6Ly9naXRodWIuY29tL1RNRWx5cmFsYWIvTXVzZVRhbGsuZ2l0IiwgIjI1MzZlYWYwNmI4NWY3NzVkOGUwNGI3ZmQ4MWJmNjY5OTE1NWE1OGQiClcgPSAiL2thZ2dsZS90ZW1wL011c2VUYWxrIgpzdWJwcm9jZXNzLnJ1bihbImdpdCIsICJjbG9uZSIsICItcSIsIFNSQywgV10sIGNoZWNrPVRydWUpCnN1YnByb2Nlc3MucnVuKFsiZ2l0IiwgIi1DIiwgVywgImNoZWNrb3V0IiwgIi1xIiwgUkVGXSwgY2hlY2s9VHJ1ZSkKcGlwID0gW3N5cy5leGVjdXRhYmxlLCAiLW0iLCAicGlwIiwgImluc3RhbGwiLCAiLXEiXQpzdWJwcm9jZXNzLnJ1bihwaXAgKyBbImRpZmZ1c2Vycz09MC4zMC4yIiwgImFjY2VsZXJhdGUiLCAiZWlub3BzIiwgIm9tZWdhY29uZiIsICJmZm1wZWctcHl0aG9uIiwgIm1vdmllcHkiLAogICAgICAgICAgICAgICAgICAgICAgInNvdW5kZmlsZSIsICJsaWJyb3NhIiwgImltYWdlaW9bZmZtcGVnXSIsICJodWdnaW5nZmFjZV9odWIiXSwgY2hlY2s9VHJ1ZSkKIyBNZWFzdXJlZCAyMDI2LTExLTI1IG9uIEthZ2dsZSAoUHl0aG9uIDMuMTMsIHRvcmNoIDIuMTEpOiBvcGVubWltIGNyYXNoZXMgb24gaW1wb3J0IChwa2d1dGlsLkltcEltcG9ydGVyIGlzIGdvbmUpCiMgYW5kIG1tY3YgMi4xIGhhcyBubyBidWlsZCBmb3IgdGhpcyBzdGFjaywgc28gbW1wb3NlL0RXUG9zZSBjYW5ub3QgYmUgaW5zdGFsbGVkLiBNdXNlVGFsayBvbmx5IHVzZXMgRFdQb3NlIHRvCiMgcmVmaW5lIHRoZSBmYWNlIGNyb3A7IHVwc3RyZWFtIGFscmVhZHkgZmFsbHMgYmFjayB0byB0aGUgUzNGRCBmYWNlIGJveCB3aGVuIGxhbmRtYXJrcyBhcmUgdW51c2FibGUuIFBhdGNoOgojIGRyb3AgdGhlIG1tcG9zZSBpbXBvcnQgKyBtb2RlbCBhbmQgYWx3YXlzIHVzZSB0aGUgZmFjZSBib3ggKGJib3hfc2hpZnQgc3RpbGwgYXBwbGllcykuIE5vIGV4dHJhIG1vZGVsLgpQUkUgPSBmIntXfS9tdXNldGFsay91dGlscy9wcmVwcm9jZXNzaW5nLnB5IgpzcmMgPSBvcGVuKFBSRSkucmVhZCgpCmZvciBsaW5lIGluICgiZnJvbSBtbXBvc2UuYXBpcyBpbXBvcnQgaW5mZXJlbmNlX3RvcGRvd24sIGluaXRfbW9kZWxcbiIsICJmcm9tIG1tcG9zZS5zdHJ1Y3R1cmVzIGltcG9ydCBtZXJnZV9kYXRhX3NhbXBsZXNcbiIsCiAgICAgICAgICAgICAibW9kZWwgPSBpbml0X21vZGVsKGNvbmZpZ19maWxlLCBjaGVja3BvaW50X2ZpbGUsIGRldmljZT1kZXZpY2UpXG4iKToKICAgIGFzc2VydCBsaW5lIGluIHNyYywgZiJNdXNlVGFsayBjaGFuZ2VkIHVwc3RyZWFtOiB7bGluZSFyfSBub3QgZm91bmQiCiAgICBzcmMgPSBzcmMucmVwbGFjZShsaW5lLCAiIikKc3JjICs9ICIiIgoKZGVmIGdldF9sYW5kbWFya19hbmRfYmJveChpbWdfbGlzdCwgdXBwZXJib25kcmFuZ2U9MCk6ICAgIyB2aWRlby1zdHVkaW8gcGF0Y2g6IFMzRkQgZmFjZSBib3gsIG5vIG1tcG9zZQogICAgZnJhbWVzID0gcmVhZF9pbWdzKGltZ19saXN0KQogICAgY29vcmRzX2xpc3QgPSBbXQogICAgZm9yIGZyIGluIGZyYW1lczoKICAgICAgICBmID0gZmEuZ2V0X2RldGVjdGlvbnNfZm9yX2JhdGNoKG5wLmFzYXJyYXkoW2ZyXSkpWzBdCiAgICAgICAgaWYgZiBpcyBOb25lOgogICAgICAgICAgICBjb29yZHNfbGlzdC5hcHBlbmQoY29vcmRfcGxhY2Vob2xkZXIpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgeDEsIHkxLCB4MiwgeTIgPSBbaW50KHYpIGZvciB2IGluIGZdCiAgICAgICAgY29vcmRzX2xpc3QuYXBwZW5kKCh4MSwgbWF4KDAsIHkxICsgaW50KHVwcGVyYm9uZHJhbmdlKSksIHgyLCB5MikpCiAgICByZXR1cm4gY29vcmRzX2xpc3QsIGZyYW1lcwoiIiIKb3BlbihQUkUsICJ3Iikud3JpdGUoc3JjKQppbXBvcnQgc2h1dGlsCmlmIG5vdCBzaHV0aWwud2hpY2goImZmbXBlZyIpOiAgICAgICAgICAgICAgICAgICMgTXVzZVRhbGsgc2hlbGxzIG91dCB0byBmZm1wZWcgZm9yIGZyYW1lcyBhbmQgdGhlIGZpbmFsIG11eAogICAgaW1wb3J0IGltYWdlaW9fZmZtcGVnCiAgICBvcy5tYWtlZGlycygiL2thZ2dsZS90ZW1wL2JpbiIsIGV4aXN0X29rPVRydWUpCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoIi9rYWdnbGUvdGVtcC9iaW4vZmZtcGVnIik6CiAgICAgICAgb3Muc3ltbGluayhpbWFnZWlvX2ZmbXBlZy5nZXRfZmZtcGVnX2V4ZSgpLCAiL2thZ2dsZS90ZW1wL2Jpbi9mZm1wZWciKQogICAgb3MuZW52aXJvblsiUEFUSCJdID0gIi9rYWdnbGUvdGVtcC9iaW46IiArIG9zLmVudmlyb25bIlBBVEgiXQpmcm9tIGh1Z2dpbmdmYWNlX2h1YiBpbXBvcnQgaGZfaHViX2Rvd25sb2FkCk0gPSBmIntXfS9tb2RlbHMiCmZvciByZXBvLCBmaWxlcywgc3ViIGluIFsKICAgICgiVE1FbHlyYWxhYi9NdXNlVGFsayIsIFsibXVzZXRhbGtWMTUvbXVzZXRhbGsuanNvbiIsICJtdXNldGFsa1YxNS91bmV0LnB0aCJdLCAiIiksCiAgICAoInN0YWJpbGl0eWFpL3NkLXZhZS1mdC1tc2UiLCBbImNvbmZpZy5qc29uIiwgImRpZmZ1c2lvbl9weXRvcmNoX21vZGVsLmJpbiJdLCAic2QtdmFlIiksCiAgICAoIm9wZW5haS93aGlzcGVyLXRpbnkiLCBbImNvbmZpZy5qc29uIiwgInB5dG9yY2hfbW9kZWwuYmluIiwgInByZXByb2Nlc3Nvcl9jb25maWcuanNvbiJdLCAid2hpc3BlciIpLApdOgogICAgZm9yIGYgaW4gZmlsZXM6CiAgICAgICAgaGZfaHViX2Rvd25sb2FkKHJlcG8sIGYsIGxvY2FsX2Rpcj1mIntNfS97c3VifSIgaWYgc3ViIGVsc2UgTSkKIyBmYWNlLXBhcnNpbmcgQmlTZU5ldCB3ZWlnaHRzIChIRiByZS11cGxvYWQsIFdURlBMOyB1cHN0cmVhbSB1c2VzIGEgR29vZ2xlIERyaXZlIGxpbmspCmltcG9ydCBvcwpvcy5tYWtlZGlycyhmIntNfS9mYWNlLXBhcnNlLWJpc2VudCIsIGV4aXN0X29rPVRydWUpCnRyeToKICAgIGhmX2h1Yl9kb3dubG9hZCgiTWFueU90aGVyRnVuY3Rpb25zL2ZhY2UtcGFyc2UtYmlzZW50IiwgIjc5OTk5X2l0ZXIucHRoIiwgbG9jYWxfZGlyPWYie019L2ZhY2UtcGFyc2UtYmlzZW50IikKZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgcmVwb3J0KGZhY2VfcGFyc2Vfd2FybmluZz1mIkJpU2VOZXQgd2VpZ2h0cyBub3QgZm91bmQgb24gSEYgKHtlfSk7IHJldmlldyBhIHNvdXJjZSBiZWZvcmUgcHJvZHVjdGlvbiB1c2UiKQoKCmRlZiBmZXRjaF92ZXJpZmllZCh1cmwsIGRzdCk6CiAgICAiIiJEb3dubG9hZCBhbiBvZmZpY2lhbCB0b3JjaC1odWIgc3R5bGUgZmlsZSAobmFtZSBlbmRzIGluIHRoZSBmaXJzdCA4IGhleCBvZiBpdHMgc2hhMjU2KSwgY2hlY2sgdGhlIGhhc2gsIGFuZAogICAgaWYgaXQgaXMgaW4gUHlUb3JjaCdzIGxlZ2FjeSAudGFyIGZvcm1hdCAocmVmdXNlZCBieSB0b3JjaD49Mi42J3Mgc2FmZSBsb2FkZXIpIHJlLXNhdmUgaXQgaW4gdGhlIGN1cnJlbnQgZm9ybWF0CiAgICDigJQgb25seSBhZnRlciB0aGUgaGFzaCBtYXRjaGVkLCBzbyB0aGUgb25lIG5vbi1zYWZlIGxvYWQgaXMgb2YgYSB2ZXJpZmllZCBvZmZpY2lhbCBmaWxlLiIiIgogICAgaW1wb3J0IGhhc2hsaWIsIHJlLCB0b3JjaAogICAgdXJsbGliLnJlcXVlc3QudXJscmV0cmlldmUodXJsLCBkc3QpCiAgICB3YW50ID0gcmUuc2VhcmNoKHIiLShbMC05YS1mXXs4fSlcLnB0aCQiLCB1cmwpLmdyb3VwKDEpCiAgICBnb3QgPSBoYXNobGliLnNoYTI1NihvcGVuKGRzdCwgInJiIikucmVhZCgpKS5oZXhkaWdlc3QoKQogICAgaWYgbm90IGdvdC5zdGFydHN3aXRoKHdhbnQpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmImhhc2ggbWlzbWF0Y2ggZm9yIHt1cmx9OiB7Z290Wzo4XX0gIT0ge3dhbnR9IikKICAgIHRyeToKICAgICAgICB0b3JjaC5sb2FkKGRzdCwgbWFwX2xvY2F0aW9uPSJjcHUiLCB3ZWlnaHRzX29ubHk9VHJ1ZSkKICAgIGV4Y2VwdCBSdW50aW1lRXJyb3IgYXMgZToKICAgICAgICBpZiAibGVnYWN5IC50YXIgZm9ybWF0IiBub3QgaW4gc3RyKGUpOgogICAgICAgICAgICByYWlzZQogICAgICAgIHRvcmNoLnNhdmUodG9yY2gubG9hZChkc3QsIG1hcF9sb2NhdGlvbj0iY3B1Iiwgd2VpZ2h0c19vbmx5PUZhbHNlKSwgZHN0KQogICAgICAgIHJlcG9ydCgqKntmImNvbnZlcnRlZF97b3MucGF0aC5iYXNlbmFtZShkc3QpfSI6ICJsZWdhY3kgLnRhciAtPiBjdXJyZW50IGZvcm1hdCAoc2hhMjU2IHZlcmlmaWVkIGZpcnN0KSJ9KQoKCiMgQmlTZU5ldCdzIGJhY2tib25lOiB0b3JjaHZpc2lvbidzIFJlc05ldC0xOCBJbWFnZU5ldCB3ZWlnaHRzIChvZmZpY2lhbCwgQlNELTMpOyBNdXNlVGFsayBleHBlY3RzIHRoaXMgZmlsZSBuYW1lCmZldGNoX3ZlcmlmaWVkKCJodHRwczovL2Rvd25sb2FkLnB5dG9yY2gub3JnL21vZGVscy9yZXNuZXQxOC01YzEwNmNkZS5wdGgiLCBmIntNfS9mYWNlLXBhcnNlLWJpc2VudC9yZXNuZXQxOC01YzEwNmNkZS5wdGgiKQojIFMzRkQgZmFjZSBkZXRlY3RvciAoZmFjZS1hbGlnbm1lbnQsIEJTRC0zKTogZmV0Y2hlZCBoZXJlIHNvIGl0IGlzIHZlcmlmaWVkLCBub3QgZG93bmxvYWRlZCBhdCBpbXBvcnQgdGltZQpmZXRjaF92ZXJpZmllZCgiaHR0cHM6Ly93d3cuYWRyaWFuYnVsYXQuY29tL2Rvd25sb2Fkcy9weXRob24tZmFuL3MzZmQtNjE5YTMxNjgxMi5wdGgiLAogICAgICAgICAgICAgICBmIntXfS9tdXNldGFsay91dGlscy9mYWNlX2RldGVjdGlvbi9kZXRlY3Rpb24vc2ZkL3MzZmQucHRoIikKaW1wb3J0IHlhbWwKY2ZnID0ge2l0WyJpZCJdOiB7InZpZGVvX3BhdGgiOiBzdHIoSU4gLyBpdFsidmlkZW8iXSksICJhdWRpb19wYXRoIjogc3RyKElOIC8gaXRbImF1ZGlvIl0pfSBmb3IgaXQgaW4gUEFSQU1TWyJpdGVtcyJdfQpvcGVuKGYie1d9L3ZzX3Rhc2sueWFtbCIsICJ3Iikud3JpdGUoeWFtbC5zYWZlX2R1bXAoY2ZnKSkKdCA9IHRpbWUudGltZSgpCnN1YnByb2Nlc3MucnVuKFtzeXMuZXhlY3V0YWJsZSwgIi1tIiwgInNjcmlwdHMuaW5mZXJlbmNlIiwgIi0taW5mZXJlbmNlX2NvbmZpZyIsICJ2c190YXNrLnlhbWwiLCAiLS1yZXN1bHRfZGlyIiwgc3RyKE9VVCksCiAgICAgICAgICAgICAgICAiLS11bmV0X21vZGVsX3BhdGgiLCAibW9kZWxzL211c2V0YWxrVjE1L3VuZXQucHRoIiwgIi0tdW5ldF9jb25maWciLCAibW9kZWxzL211c2V0YWxrVjE1L211c2V0YWxrLmpzb24iLAogICAgICAgICAgICAgICAgIi0tdmVyc2lvbiIsICJ2MTUiLCAqKFsiLS11c2VfZmxvYXQxNiJdIGlmIFBBUkFNUy5nZXQoImZwMTYiLCBUcnVlKSBlbHNlIFtdKV0sIGN3ZD1XLCBjaGVjaz1UcnVlKQpyZXBvcnQobGlwc3luY19zPXJvdW5kKHRpbWUudGltZSgpIC0gdCwgMSksIG91dHB1dHM9W3N0cihwLnJlbGF0aXZlX3RvKE9VVCkpIGZvciBwIGluIE9VVC5yZ2xvYigiKi5tcDQiKV0sIG11c2V0YWxrX3JlZj1SRUYsCiAgICAgICBmYWNlX2Nyb3A9IlMzRkQgZmFjZSBib3ggKERXUG9zZSBsYW5kbWFya3MgdW5hdmFpbGFibGUgb24gdGhpcyBpbWFnZSkiKQo=\'\nexec(base64.b64decode(_COMMON_B64).decode())\n\n\ndef _env_check():  # did Kaggle grant what the job asked for? (GPU + internet need a phone-verified account)\n    import shutil as _sh, socket as _so, subprocess as _sp\n    gpu = bool(_sh.which("nvidia-smi")) and _sp.run(["nvidia-smi", "-L"], capture_output=True).returncode == 0\n    try:\n        _so.getaddrinfo("huggingface.co", 443); net = True\n    except OSError:\n        net = False\n    report(gpu_granted=gpu, internet_granted=net)\n    if PARAMS.get("_need_gpu") and not gpu:\n        report(error=("Kaggle gave this session no GPU: this accelerator is busy/unavailable or another GPU session "\n                      "is running (free accounts get one at a time); retry later.") if net else\n                     ("Kaggle gave this session no GPU and no internet: the account needs phone verification "\n                      "(kaggle.com > Settings > Phone verification)."), gpu_unavailable_reason="busy" if net else "unverified")\n        raise SystemExit(3)\n    if PARAMS.get("_need_internet") and not net:\n        report(error="Kaggle gave this session no internet (phone verification needed, or internet off).")\n        raise SystemExit(3)\n\n\n_env_check()\ntry:\n    import torch as _t\n    if _t.cuda.is_available():\n        report(gpu=gpu_info())\nexcept Exception:\n    pass\ntry:\n    exec(compile(base64.b64decode(_JOB_B64).decode(), "job.py", "exec"))\nexcept _Done:\n    pass\nif json.loads((OUT / "report.json").read_text()).get("error"):\n    raise SystemExit(4)\n'
open('/content/vs_job.py', 'w').write(JOB)
import subprocess, sys
rc = subprocess.run([sys.executable, '/content/vs_job.py']).returncode
print('job exit code', rc, '(0 = ok; see report.json in the zip either way)')


In [ ]:
# 4. pack the results and download them
import json, pathlib, shutil
out = pathlib.Path('/kaggle/working/out')
rep = out / 'report.json'
if rep.exists():
    d = json.loads(rep.read_text()); d['provider'] = 'colab'; rep.write_text(json.dumps(d, indent=2))
    print(json.dumps(d, indent=1)[:3000])
zp = shutil.make_archive('/content/vs-result-musetalk', 'zip', out)
from google.colab import files
files.download(zp)


In [ ]:
# 5. release the GPU (run after the download has started)
from google.colab import runtime
runtime.unassign()
